In [0]:
%sql
/*
Your target table already contains records loaded until
2026-11-11
.
You can use both SQL and Apache Spark for incremental loading—they aren't mutually exclusive, .
When to Use SQL
SQL is ideal when the source and target tables already live inside a relational database or cloud data warehouse (such as PostgreSQL, MySQL, SQL Server, Snowflake, BigQuery, or Redshift).
*/

MERGE INTO target_customer AS T
USING staging_customer AS S
ON T.customer_id = S.customer_id

WHEN MATCHED THEN UPDATE SET
    T.name = S.name,
    T.city = S.city,
    T.updated_at = S.updated_at

WHEN NOT MATCHED THEN INSERT (
    customer_id, name, city, updated_at
)
VALUES (
    S.customer_id, S.name, S.city, S.updated_at
);


In [0]:
%sql

/* 
1. Create the Metadata Table

Design a table that tracks one row per pipeline or table name so it can scale across multiple jobs.
*/

CREATE TABLE dbo.etl_watermark (
    table_name VARCHAR(100) PRIMARY KEY,
    last_watermark DATETIME2 NOT NULL,
    last_run_at DATETIME2 DEFAULT SYSUTCDATETIME()
);

-- Seed an initial baseline watermark for your table
INSERT INTO dbo.etl_watermark (table_name, last_watermark)
VALUES ('customers', '1900-01-01 00:00:00');


In [0]:
%sql
/*
 Full Incremental Load Pattern with Watermark Update

The safest pattern captures the new maximum watermark from the source batch into a variable (or staging table) before merging, then updates the metadata table inside the same transaction.
*/

BEGIN TRANSACTION;

BEGIN TRY
    DECLARE @LastWatermark DATETIME2;
    DECLARE @NewWatermark DATETIME2;

    -- Step 1: Read the current watermark from the metadata table
    SELECT @LastWatermark = last_watermark
    FROM dbo.etl_watermark
    WHERE table_name = 'customers';

    -- Step 2: Stage the delta rows and capture the new max timestamp
    SELECT customer_id, full_name, email, updated_at
    INTO #staged_customers
    FROM dbo.source_customers
    WHERE updated_at > @LastWatermark;

    SELECT @NewWatermark = MAX(updated_at)
    FROM #staged_customers;

    -- Step 3: Only run MERGE and update watermark if new rows exist
    IF @NewWatermark IS NOT NULL
    BEGIN
        MERGE INTO dbo.target_customers AS target
        USING #staged_customers AS source
        ON target.customer_id = source.customer_id
        WHEN MATCHED THEN
            UPDATE SET 
                target.full_name  = source.full_name,
                target.email      = source.email,
                target.updated_at = source.updated_at
        WHEN NOT MATCHED BY TARGET THEN
            INSERT (customer_id, full_name, email, updated_at)
            VALUES (source.customer_id, source.full_name, source.email, source.updated_at);

        -- Step 4: Save the new watermark to the metadata table
        UPDATE dbo.etl_watermark
        SET last_watermark = @NewWatermark,
            last_run_at = SYSUTCDATETIME()
        WHERE table_name = 'customers';
    END

    DROP TABLE #staged_customers;
    COMMIT TRANSACTION;
END TRY
BEGIN CATCH
    IF @@TRANCOUNT > 0
        ROLLBACK TRANSACTION;
    THROW;
END CATCH;
